# 19. Resultados adicionais e figuras

Desenvolve `app/graficos.py` (as nove figuras dos resultados) e cobre os campos
novos da esteira: `A_t`, as médias `trajetoria_V_media` e
`trajetoria_u_media` e os percentis por período. **F11, F15.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from app import nucleo
from app.principal import executar_pipeline
from app.mercado import RendaVariavel

## Desenvolvimento

As funções abaixo foram escritas aqui e, após os testes, movidas para `app/graficos.py`.

In [ ]:
DESTINO_PADRAO = "results"

# os valores de gamma do grafico de sensibilidade. Cada ponto refaz a otimizacao.
GRADE_GAMMA = (1.5, 2.0, 3.0, 5.0, 8.0, 10.0, 15.0, 20.0)

# os graficos de mu e de sigma refazem a otimizacao nos mesmos cenarios.
# O de theta_0 usa esses beta anuais.
DESLOCAMENTOS_MU_ANUAL = (-0.06, -0.04, -0.02, 0.0, 0.02, 0.04, 0.06)
ESCALAS_SIGMA = (0.5, 0.75, 1.0, 1.25, 1.5)
BETAS_ANUAIS = (0.86, 0.88, 0.90, 0.92, 0.94, 0.96, 0.98)

# o grafico do consumo contra o premio de risco usa esses premios (mu - R_f,
# ao ano) e esses valores de gamma
PREMIOS_ANUAIS = (0.0, 0.02, 0.04, 0.06, 0.08)
GAMMAS_CONSUMO = (0.5, 1.0, 2.0, 5.0)

In [3]:
def _rodape(fig, texto: str) -> None:
    """Escreve as duas linhas de informacoes no rodape da figura."""
    fig.text(0.5, 0.012, texto, ha="center", va="bottom", fontsize=6.5,
             color="0.45", linespacing=1.5)
    fig.subplots_adjust(bottom=0.26)

In [4]:
def _salvar(fig, destino: str, nome: str, rodape: str) -> str:
    _rodape(fig, rodape)
    caminho = os.path.join(destino, nome)
    fig.savefig(caminho, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return caminho

In [ ]:
def montar_rodape(res: dict, cfg: dict, periodo: tuple[str, str], n_obs: int,
                  beta_anual: float, anos: float, unidade: str,
                  dados_reais: bool = True) -> str:
    """Texto de procedencia impresso em todas as figuras."""
    rf_anual = (1.0 + res["rf"]) ** cfg["periodos_por_ano"] - 1.0
    origem_rf = "informado" if cfg.get("cdi_anual") is not None else "série CDI"
    fonte = "dados reais" if dados_reais else "dados SINTÉTICOS"
    cenarios = f"{cfg['n_scenarios']:,}".replace(",", ".")
    trajetorias = f"{cfg['n_paths']:,}".replace(",", ".")
    periodo_beta = "pregão" if cfg["periodos_por_ano"] == 252 else "mês"
    return (f"Ibovespa {unidade} ({fonte}) · {periodo[0]} a {periodo[1]} "
            f"({n_obs} obs) · R_f={rf_anual:.2%} a.a. ({origem_rf})\n"
            f"γ={cfg['gamma']:g} · β={beta_anual:g} a.a. "
            f"({res['beta']:.6f} por {periodo_beta}) · T={anos:g} anos · "
            f"W₀={cfg['w0']:g} · {cenarios} cenários · "
            f"{trajetorias} trajetórias · seed {cfg['seed']} · "
            f"α*={res['alpha_star'][0]:.4f}")

In [ ]:
def gerar(res: dict, mercado, rf: float, cfg: dict, rodape: str,
          destino: str = DESTINO_PADRAO) -> list[str]:
    """Faz as nove figuras e devolve os caminhos dos arquivos escritos.

    O res e o que o executar_pipeline devolveu. O mercado e o rf so sao
    necessarios para os cinco graficos que voltam a usar os cenarios: quatro
    refazem a otimizacao e o do G(alpha) so reavalia a FOC. O consumo somado
    por ano ja vem pronto em res["consumo_por_ano"], entao a figura e a linha
    de comando leem o mesmo numero.
    """
    os.makedirs(destino, exist_ok=True)
    g = float(cfg["gamma"])
    Rf = 1.0 + rf
    T = res["horizonte"]
    escritos = []

    ppa = cfg["periodos_por_ano"]
    r = mercado.amostrar(cfg["n_scenarios"], seed=cfg["seed"])
    R = np.maximum(1.0 + r, 0.0)

    # 1. G(alpha) contra alpha, marcando onde cruza o zero
    a_star = float(res["alpha_star"][0])
    grade = np.linspace(a_star - 1.0, a_star + 1.0, 60)
    G = [nucleo.funcao_foc(np.array([a]), R, Rf, g)[0] for a in grade]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axhline(0, color="0.7", lw=0.8)
    ax.plot(grade, G, color="#1f77b4")
    ax.plot([a_star], [0], "o", color="#d62728", zorder=5,
            label=f"α* = {a_star:.4f}")
    ax.set_xlabel("α"); ax.set_ylabel("G(α)")
    ax.set_title("Condição de primeira ordem: G(α) = 0")
    ax.legend()
    escritos.append(_salvar(fig, destino, "foc_G_de_alpha.png", rodape))

    # 2. alpha contra gamma
    alphas = [nucleo.resolver_alpha_otimo(R, Rf, gi)[0] for gi in GRADE_GAMMA]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(GRADE_GAMMA, alphas, "o-", color="#1f77b4")
    ax.axvline(g, color="0.7", ls="--", lw=0.8)
    ax.set_xlabel("γ (aversão relativa ao risco)"); ax.set_ylabel("α*")
    ax.set_title("Sensibilidade da carteira ótima à aversão ao risco")
    escritos.append(_salvar(fig, destino, "alpha_vs_gamma.png", rodape))

    # 3. alpha contra mu
    mu = float(res["mu_hat"][0])
    mu_anual = (1.0 + mu) ** ppa - 1.0
    medias = mu_anual + np.array(DESLOCAMENTOS_MU_ANUAL)
    alphas_mu = [nucleo.resolver_alpha_otimo(
                     np.maximum(1.0 + r + (1.0 + m) ** (1.0 / ppa) - 1.0 - mu, 0.0), Rf, g)[0]
                 for m in medias]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axhline(0, color="0.7", lw=0.8)
    ax.plot(100 * medias, alphas_mu, "o-", color="#1f77b4")
    ax.axvline(100 * mu_anual, color="0.7", ls="--", lw=0.8)
    ax.set_xlabel(r"$\mu$ (retorno esperado, % a.a.)"); ax.set_ylabel(r"$\alpha^*$")
    ax.set_title("Sensibilidade da carteira ótima ao retorno esperado")
    escritos.append(_salvar(fig, destino, "alpha_vs_mu.png", rodape))

    # 4. alpha contra sigma
    media_r = r.mean(axis=0)
    sigma_anual = float(np.sqrt(res["sigma_hat"][0, 0] * ppa))
    alphas_sigma = [nucleo.resolver_alpha_otimo(
                        np.maximum(1.0 + media_r + (r - media_r) * k, 0.0), Rf, g)[0]
                    for k in ESCALAS_SIGMA]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(100 * sigma_anual * np.array(ESCALAS_SIGMA), alphas_sigma, "o-", color="#1f77b4")
    ax.axvline(100 * sigma_anual, color="0.7", ls="--", lw=0.8)
    ax.set_xlabel(r"$\sigma$ (volatilidade, % a.a.)"); ax.set_ylabel(r"$\alpha^*$")
    ax.set_title("Sensibilidade da carteira ótima à volatilidade")
    escritos.append(_salvar(fig, destino, "alpha_vs_sigma.png", rodape))

    # 5. as fracoes de consumo ao longo do tempo (F13)
    theta = res["theta"]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(theta)), theta, color="#1f77b4")
    ax.set_yscale("log")          # sem log, o theta_t fica rente a zero e so dispara no fim
    ax.set_xlabel("t (períodos)"); ax.set_ylabel(r"$\theta_t$ (escala log)")
    ax.set_title(r"Fração de consumo $\theta_t$ (crescente até $\theta_T = 1$)")
    escritos.append(_salvar(fig, destino, "theta_t.png", rodape))

    # 6. theta_0 contra beta.
    theta_0 = [nucleo.fracoes_consumo(
                   nucleo.recorrencia_A(res["phi_hat"], b ** (1.0 / ppa), g, T), g)[0]
               for b in BETAS_ANUAIS]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(BETAS_ANUAIS, theta_0, "o-", color="#1f77b4")
    ax.axvline(res["beta"] ** ppa, color="0.7", ls="--", lw=0.8)
    ax.set_xlabel(r"$\beta$ (fator de desconto, ao ano)")
    ax.set_ylabel(r"$\theta_0$ (fração consumida em t = 0)")
    ax.set_title("Sensibilidade da fração de consumo ao fator de desconto")
    escritos.append(_salvar(fig, destino, "theta_vs_beta.png", rodape))

    # 7. theta_0 contra o premio de risco, uma curva por gamma.
    rf_anual = (1.0 + rf) ** ppa - 1.0
    media_amostra = float(r.mean())
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.axhline(0, color="0.7", lw=0.8)
    for gi in GAMMAS_CONSUMO:
        theta_p = []
        for p in PREMIOS_ANUAIS:
            cenarios = np.maximum(1.0 + r + (1.0 + rf_anual + p) ** (1.0 / ppa) - 1.0 - media_amostra, 0.0)
            a = nucleo.resolver_alpha_otimo(cenarios, Rf, gi)
            A_p = nucleo.recorrencia_A(nucleo.phi_chapeu(a, cenarios, Rf, gi), res["beta"], gi, T)
            theta_p.append(nucleo.fracoes_consumo(A_p, gi)[0])
        ax.plot(100 * np.array(PREMIOS_ANUAIS), 100 * (np.array(theta_p) / theta_p[0] - 1.0),
                "o-", label=rf"$\gamma$ = {gi:g}")
    ax.set_xlabel(r"prêmio de risco $\mu - R_f$ (p.p. ao ano)")
    ax.set_ylabel(r"variação de $\theta_0$ desde o prêmio zero (%)", fontsize=9)
    ax.set_title("Efeito do prêmio de risco sobre a fração de consumo")
    ax.legend()
    escritos.append(_salvar(fig, destino, "theta_vs_premio.png", rodape))

    # 8. a riqueza com a faixa entre os percentis 5 e 95
    t = np.arange(T + 1)
    media, p5, p95 = (res["trajetoria_W_media"], res["trajetoria_W_p5"],
                      res["trajetoria_W_p95"])
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7, 5.4), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1]})
    ax.fill_between(t, p5, p95, color="#1f77b4", alpha=0.3, label="P5-P95")
    ax.plot(t, media, color="#1f77b4", label="média")
    ax.set_yscale("log")
    ax.set_ylabel(r"$W_t$ (escala log)")
    ax.set_title("Trajetória da riqueza")
    ax.legend()

    largura = 100.0 * (p95 - p5) / np.where(media > 0, media, np.nan)
    ax2.plot(t, largura, color="#7f7f7f")
    ax2.set_xlabel("t (períodos)")
    ax2.set_ylabel("P95-P5\n(% da média)", fontsize=8)
    escritos.append(_salvar(fig, destino, "riqueza_W_t.png", rodape))

    # 9. consumo somado por ano (sem o c_T, que e a liquidacao terminal)
    por_ano = res["consumo_por_ano"]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(np.arange(1, len(por_ano) + 1), por_ano, color="#ff7f0e")
    ax.set_xlabel("ano"); ax.set_ylabel(r"consumo (fração de $W_0$)")
    ax.set_title("Consumo agregado por ano")
    escritos.append(_salvar(fig, destino, "consumo_por_ano.png", rodape))

    return escritos

**Teste**: resultado base sobre o qual as figuras são desenhadas.

In [7]:
ret = pd.DataFrame({'data': pd.bdate_range('2022-05-24', periods=400).strftime('%Y-%m-%d'),
                    'ibov': 0.0007 + np.random.default_rng(0).normal(0, 0.011, 400),
                    'cdi': np.full(400, 0.00049)})
res = executar_pipeline({'retornos': ret, 'ativos': ['ibov'], 'periodos_por_ano': 252,
                         'gamma': 5.0, 'beta_anual': 0.96, 'w0': 1.0, 'horizonte': 252,
                         'n_scenarios': 200_000, 'n_paths': 500, 'seed': 1})
mkt = RendaVariavel(ret[['data', 'ibov']])
T = res['horizonte']

print('alpha* =', res['alpha_star'][0], '| T =', T)

alpha* = -0.36744844636966967 | T = 252


**Teste**: Etapas 3 e 7: `A_t` e a média da função valor V_t(W_t) nas trajetórias (F11).

In [8]:
print('A_0 =', res['A_t'][0], '| A_T =', res['A_t'][-1],
      '| E[V_0] =', res['trajetoria_V_media'][0], '| E[V_T] =', res['trajetoria_V_media'][-1])

A_0 = 779604271647.1617 | A_T = 1.0 | E[V_0] = -194901067911.79153 | E[V_T] = -762581895.7626461


In [9]:
assert res['A_t'].shape == (T + 1,) and res['trajetoria_V_media'].shape == (T + 1,)
assert np.isclose(res['A_t'][-1], 1.0)
# em t=0 todos os caminhos tem W_0 = 1, entao a media e o proprio V_0(W_0)
assert np.isclose(res['trajetoria_V_media'][0], nucleo.funcao_valor(res['A_t'][:1], 1.0, 5.0)[0])
# em t=T a funcao valor e a utilidade da riqueza terminal, que e toda consumida
assert np.allclose(res['trajetoria_V_media'][-1], res['trajetoria_u_media'][-1])
assert np.allclose(res['theta'], res['A_t'] ** (-1 / 5.0))

**Teste**: percentis por período, que sustentam a faixa entre os percentis 5 e 95.

In [10]:
p5, p50, p95 = res['trajetoria_W_p5'], res['trajetoria_W_mediana'], res['trajetoria_W_p95']

print('em t=T:  P5=%.6f  mediana=%.6f  P95=%.6f' % (p5[-1], p50[-1], p95[-1]))

em t=T:  P5=0.003878  mediana=0.004294  P95=0.004742


In [11]:
for k in ('trajetoria_W_p5', 'trajetoria_W_mediana', 'trajetoria_W_p95'):
    assert res[k].shape == (T + 1,), k
assert np.all(p5 <= p50) and np.all(p50 <= p95)
assert np.allclose([p5[0], p50[0], p95[0]], 1.0)
assert p95[-1] - p5[-1] > 0

**Teste**: o rodapé de procedência impresso em cada figura.

In [12]:
cfg = {'gamma': 5.0, 'n_scenarios': 50_000, 'seed': 1,
       'periodos_por_ano': 252, 'w0': 1.0, 'n_paths': 500}
rodape = montar_rodape(res, cfg, ('2022-05-24', '2024-01-01'), len(ret), 0.96, 1.0, 'diario')

print(rodape)

Ibovespa diario (dados reais) · 2022-05-24 a 2024-01-01 (400 obs) · R_f=13.14% a.a. (série CDI)
γ=5 · β=0.96 a.a. (0.999838 por pregão) · T=1 anos · W₀=1 · 50.000 cenários · 500 trajetórias · seed 1 · α*=-0.3674


In [13]:
assert 'γ=5' in rodape and 'seed 1' in rodape and 'α*=' in rodape and '2022-05-24' in rodape
assert len(rodape.splitlines()) == 2
assert 'R_f=' in rodape and 'W₀=' in rodape and 'trajetórias' in rodape
assert 'β=0.96 a.a.' in rodape and 'por pregão' in rodape
assert max(len(l) for l in rodape.splitlines()) < 152

**Teste**: as combinações de procedência — R_f da série ou informado, base real ou sintética.

In [14]:
res_flag = executar_pipeline({'retornos': ret, 'ativos': ['ibov'], 'periodos_por_ano': 252,
                              'cdi_anual': 0.08, 'gamma': 5.0, 'beta_anual': 0.96, 'w0': 1.0,
                              'horizonte': 252, 'n_scenarios': 200_000, 'n_paths': 500, 'seed': 1})
cfg_flag = {**cfg, 'cdi_anual': 0.08}

In [15]:
def _linha1(r, c, reais):
    """So a primeira linha do rodape, que e a da procedencia."""
    return montar_rodape(r, c, ('2022-05-24', '2024-01-01'), len(ret),
                         0.96, 1.0, 'diario', dados_reais=reais).splitlines()[0]

In [16]:
serie_real = _linha1(res,      cfg,      True)
flag_real  = _linha1(res_flag, cfg_flag, True)
serie_sint = _linha1(res,      cfg,      False)

In [17]:
for nome, txt in [('serie + real', serie_real), ('informado + real', flag_real),
                  ('serie + sintetico', serie_sint)]:
    print(f'{nome:>18}: {txt}')
print()
print('alpha* com R_f da serie :', round(res['alpha_star'][0], 4))
print('alpha* com R_f de 8% a.a.:', round(res_flag['alpha_star'][0], 4))

      serie + real: Ibovespa diario (dados reais) · 2022-05-24 a 2024-01-01 (400 obs) · R_f=13.14% a.a. (série CDI)
  informado + real: Ibovespa diario (dados reais) · 2022-05-24 a 2024-01-01 (400 obs) · R_f=8.00% a.a. (informado)
 serie + sintetico: Ibovespa diario (dados SINTÉTICOS) · 2022-05-24 a 2024-01-01 (400 obs) · R_f=13.14% a.a. (série CDI)

alpha* com R_f da serie : -0.3674
alpha* com R_f de 8% a.a.: -0.0592


In [18]:
assert '(série CDI)' in serie_real and '(informado)' in flag_real
assert 'R_f=8.00% a.a.' in flag_real
assert '(dados reais)' in serie_real and '(dados SINTÉTICOS)' in serie_sint
assert res_flag['alpha_star'][0] > res['alpha_star'][0]

**Teste**: geração das nove figuras.

In [19]:
# Gera num diretorio temporario para nao tocar em results/.
import shutil

destino = os.path.join(tempfile.gettempdir(), 'graficos_dev19')
shutil.rmtree(destino, ignore_errors=True)
escritos = gerar(res, mkt, res['rf'], cfg, rodape, destino=destino)

print('figuras:', [os.path.basename(c) for c in escritos])

figuras: ['foc_G_de_alpha.png', 'alpha_vs_gamma.png', 'alpha_vs_mu.png', 'alpha_vs_sigma.png', 'theta_t.png', 'theta_vs_beta.png', 'theta_vs_premio.png', 'riqueza_W_t.png', 'consumo_por_ano.png']


In [20]:
assert len(escritos) == 9
assert {'alpha_vs_mu.png', 'alpha_vs_sigma.png', 'theta_vs_beta.png', 'theta_vs_premio.png'} <= {os.path.basename(c) for c in escritos}

In [21]:
for c in escritos:
    assert os.path.exists(c) and os.path.getsize(c) > 5_000

In [22]:
assert sorted(os.listdir(destino)) == sorted(os.path.basename(c) for c in escritos)
assert os.path.abspath(destino) != os.path.abspath(DESTINO_PADRAO)

**Teste**: o `app.principal` não pode arrastar o matplotlib junto.

In [23]:
import subprocess

In [24]:
saida = subprocess.run(
    [sys.executable, '-c', "import sys; import app.principal; print('matplotlib' in sys.modules)"],
    capture_output=True, text=True, cwd=RAIZ).stdout.strip()

print('app.principal carrega matplotlib?', saida)

app.principal carrega matplotlib? False


In [25]:
assert saida == 'False'